#Lecture 3: Datasets continued, visualizing data, time series data

1. Orbis continued
2. Visualizing data
3. Introduction to time series data and collecting time series data from Orbis

###Start with the same dataset as for the previous lecture


In [ ]:
import pandas as pd
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
df = pd.read_excel("/content/hotel_restaurant_big_data.xlsx", index_col=0)
df

In [ ]:
df.columns

In [ ]:
#Fast-forwards same cleaning steps as before
#remove duplicates
df.drop_duplicates(inplace = True)
#Fix missing values
df.replace({"n.a.": None, "n.s.": None}, inplace=True)
#numeric columns
num_cols = [ 'Operating revenue (Turnover) EUR Last avail. yr',
       'Number of employees Last avail. yr', 'Total assets EUR Last avail. yr',
       'Profit margin Last avail. yr',
       'ROE using P/L before tax Last avail. yr',
       'ROA using Net income Last avail. yr', 'Current ratio Last avail. yr',
       'Long term debt EUR Last avail. yr',
       'Loans & short-term debt EUR Last avail. yr']

df[num_cols] = df[num_cols].astype('float64')
#Drop missing values
df.dropna(inplace=True)
#Only keep 2025 observations
df = df[df['Last avail. year']==2025]
#Drop some columns we don't need
df = df.drop(columns=[ 'Quoted', 'Branch',
       'OwnData', 'Woco', 'Country ISO code'])

df

In [ ]:
#Now check that everything is correct
df.info()

In [ ]:
df.describe()

#Moving on to visualizing the data

To analyze the two industries we need to shorten the industry code

In [ ]:
#There are many ways to do this but one example
#55=hotel 56=restaurant

df['short_industry'] = df["NACE Rev. 2, core code (4 digits)"].astype(str).str[:2].astype(int)
df["Industry"] = df["short_industry"].map({55: "Hotel", 56: "Restaurant"})
df[["NACE Rev. 2, core code (4 digits)", 'short_industry', "Industry"]]

Start by winsorizing the data and calculating some new variables

In [ ]:
df[num_cols] = df[num_cols].clip(lower=df[num_cols].quantile(0.025), upper=df[num_cols].quantile(0.975), axis=1)

In [ ]:
import numpy as np

In [ ]:
#Logarithm of total assets
df["log_assets"] = np.log1p(df["Total assets EUR Last avail. yr"])
df["log_revenue"] = np.log1p(df["Operating revenue (Turnover) EUR Last avail. yr"])

In [ ]:
#A new leverage variable total debt/total assets
df["leverage"] = (df["Loans & short-term debt EUR Last avail. yr"]+df["Long term debt EUR Last avail. yr"])/df["Total assets EUR Last avail. yr"]*100

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [ ]:
df[num_cols + ["log_assets", "log_revenue", "leverage"]].hist(figsize=(10,10),grid=False,edgecolor='k')
plt.show()

Similar plots but now we split the data by industry

In [ ]:
df.value_counts("Industry")

In [ ]:
df.boxplot(
    column='log_assets',
    by='Industry'
)

plt.xlabel('Industry')
plt.ylabel('Log_Assets')
plt.title('Assets by industry')
plt.suptitle('')
plt.show()

How do you interpret this result? Does it make sense?

In [ ]:
df.boxplot(
    column='log_revenue',
    by='Industry'
)

plt.xlabel('Industry')
plt.ylabel('Log_Revenue')
plt.title('Revenue by industry')
plt.suptitle('')
plt.show()

In [ ]:
df.boxplot(
    column='leverage',
    by="Industry"
)

plt.xlabel('Industry')
plt.ylabel('Leverage (Debt / Assets)')
plt.title('Leverage by Industry')
plt.show()

In [ ]:
for industry in df['Industry'].unique():
    subset = df[df['Industry'] == industry]
    plt.hist(
        subset['log_revenue'].dropna(),
        alpha=0.5,
        label=f'Industry {industry}'
    )

plt.xlabel('Revenue')
plt.ylabel('Frequency')
plt.title('Revenue distribution by industry')
plt.legend()
plt.show()

In [ ]:
sns.scatterplot(
    data=df,
    x='log_assets',
    y='ROA using Net income Last avail. yr',
    hue='Industry'
)

In [ ]:
sns.scatterplot(
    data=df,
    x='leverage',
    y='Current ratio Last avail. yr',
    hue='Industry'
)

Comparing industry means for multiple variables

In [ ]:
variables = [
    'leverage',
    'ROA using Net income Last avail. yr',
    'ROE using P/L before tax Last avail. yr'
]
means_df = df.groupby('Industry')[variables].mean()

In [ ]:
ax = means_df[variables].plot.bar(
    figsize=(5, 5),
    grid=True,
    colormap='viridis',
    edgecolor='black'
)
ax.set_title('Hotel and Restaurant financial ratios')
ax.set_ylabel('%')
ax.set_xticklabels(["Hotels", "Restaurants"], rotation=30, ha='right')
#ax.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
plt.legend([    'leverage',
    'ROA using Net income Last avail. yr',
    'ROE using P/L before tax Last avail. yr'])
plt.tight_layout()
plt.show()

#Time series data

* New dataset
* Visualization continued

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
ts_df = pd.read_excel("/content/time_series_lecture.xlsx", index_col=0)
ts_df

In [ ]:
ts_df.columns

In [ ]:
id_cols = [
    'Company name Latin alphabet',
    'Country ISO code',
    'NACE Rev. 2, core code (4 digits)',
    'Consolidation code'
]

variables = {
    "ROA": "ROA using Profit (Loss) before tax",
    "Revenue": "Operating revenue (Turnover) EUR",
    "Assets": "Total assets EUR",
    "CurrentRatio": "Current ratio",
    "LongTermDebt": "Long term debt EUR",
    "ShortTermDebt": "Loans & short-term debt EUR",
    "AccountsReceivable": "Debtors EUR",
    "AccountsPayable": "Creditors EUR",
    "Inventory": "Stock EUR",
    "COGS": "Costs of goods sold EUR"
}

panel_list = []

for year in range(2017, 2023):

    temp = ts_df[id_cols].copy()

    temp["Year"] = year

    for short_name, prefix in variables.items():

        col = f"{prefix} {year}"

        if col in ts_df.columns:
            temp[short_name] = ts_df[col]

    panel_list.append(temp)

panel_df = pd.concat(panel_list, ignore_index=True)

panel_df

In [ ]:
#Fast-forwards cleaning steps
#remove duplicates
panel_df.drop_duplicates(inplace = True)
#Fix missing values
panel_df.replace({"n.a.": None, "n.s.": None}, inplace=True)
#numeric columns
num_cols = [  'ROA', 'Revenue', 'Assets', 'CurrentRatio', 'LongTermDebt',
       'ShortTermDebt', 'Debtors', 'AccountsPayable', 'AccountsReceivable',
       'COGS']

panel_df[num_cols] = panel_df[num_cols].astype('float64')
#Drop missing values
panel_df.dropna(inplace=True)

panel_df

##Calculate variables needed for our analysis

* Revenue change
* Log assets
* Leverage
* Cash conversion cycle
* Accounts payable turnover
* Inventory turnover
* Accounts receivable turnover

Already in the data
* Current ratio
* ROA

In [ ]:
panel_df[num_cols].corr()

In [ ]:
corr = panel_df[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
yearly_avg = (
    panel_df.groupby("Year")[num_cols]
    .mean()
)

yearly_avg.plot(
    figsize=(12,6),
    marker="o"
)

plt.title("Average Financial Metrics by Year")
plt.ylabel("Value")
plt.grid(True)
plt.show()

In [ ]:
panel_df.groupby("Year")[num_cols].mean()

#What if we standardize the data?

It is difficult to plot variables with very different scales in a same plot unless the data is standardized

Z-scores: mean is equal to 0 and the value of one standard deviation is 1, all values range between -3 and 3

In [ ]:
from sklearn.preprocessing import StandardScaler

yearly_avg = (
    panel_df.groupby("Year")[num_cols]
    .mean()
)

scaled = pd.DataFrame(
    StandardScaler().fit_transform(yearly_avg),
    columns=yearly_avg.columns,
    index=yearly_avg.index
)

scaled.plot(
    figsize=(12,6),
    marker="o"
)

plt.title("Standardized Financial Trends")
plt.ylabel("Z-score")
plt.grid(True)
plt.show()